In [81]:
def agent_minimax(obs, config):
    import numpy as np
    import random
    def is_terminal_window(window, config):
        return window.count(1) == config.inarow or window.count(2) == config.inarow
    def is_terminal_node(grid, config):
        if list(grid[0, :]).count(0) == 0:
            return True
        for row in range(config.rows):
            for col in range(config.columns-(config.inarow-1)):
                window = list(grid[row, col:col+config.inarow])
                if is_terminal_window(window, config):
                    return True
        for row in range(config.rows-(config.inarow-1)):
            for col in range(config.columns):
                window = list(grid[row:row+config.inarow, col])
                if is_terminal_window(window, config):
                    return True
        for row in range(config.rows-(config.inarow-1)):
            for col in range(config.columns-(config.inarow-1)):
                window = list(grid[range(row, row+config.inarow), range(col, col+config.inarow)])
                if is_terminal_window(window, config):
                    return True
        for row in range(config.inarow-1, config.rows):
            for col in range(config.columns-(config.inarow-1)):
                window = list(grid[range(row, row-config.inarow, -1), range(col, col+config.inarow)])
                if is_terminal_window(window, config):
                    return True
        return False

    def check_window(window,num_discs, piece):
        return (window.count(piece) == num_discs)
    def score_board(board, configuration, mark):
        score = 0
        opponent_mark = 2 if mark==1 else 1
        for k in range(2, configuration.inarow+1):
            for i in range(configuration.rows):
                for j in range(configuration.columns - k + 1):
                    score += check_window(list(board[i, j:j+k]), k, mark)*(1000^(k-1))
                    score -= check_window(list(board[i, j:j+k]), k, opponent_mark)*(2000^(k-1))
            for i in range(configuration.rows - k + 1):
                for j in range(configuration.columns):
                    score += check_window(list(board[i:i+k, j]), k, mark)*(1000^(k-1))
                    score -= check_window(list(board[i:i+k, j]), k, opponent_mark)*(2000^(k-1))
            for i in range(configuration.rows - k + 1):
                for j in range(configuration.columns - k + 1):
                    score += check_window(list(board[i:i+k, j:j+k].diagonal()), k, mark)*(1000^(k-1))
                    score -= check_window(list(board[i:i+k, j:j+k].diagonal()), k, opponent_mark)*(2000^(k-1))
                    score += check_window(list(np.flipud(board[i:i+k, j:j+k]).diagonal()), k, mark)*(1000^(k-1))
                    score -= check_window(list(np.flipud(board[i:i+k, j:j+k]).diagonal()), k, opponent_mark)*(2000^(k-1))
        return score

    def drop_piece(grid, col, piece, config):
        next_grid = grid.copy()
        for row in range(config.rows-1, -1, -1):
            if next_grid[row][col] == 0:
                break
        next_grid[row][col] = piece
        return next_grid

    def minimax(board, depth, maximizing, mark, config, alpha, beta):
        opponent_mark = 2 if mark==1 else 1
        valid_moves = [col for col in range(config.columns) if board[0][col] == 0]
        if depth==0 or is_terminal_node(board, config):
            return score_board(board, config, mark)
        if maximizing:
            value = -np.inf
            for col in valid_moves:
                new_board = drop_piece(board, col, mark, config)
                value = np.maximum(value, minimax(new_board, depth-1, False, mark, config, alpha, beta))
                if value >= beta:
                    break
                alpha = np.maximum(alpha, value)
            return value
        else:
            value = np.inf
            for col in valid_moves:
                new_board = drop_piece(board, col, opponent_mark, config)
                value = np.minimum(value, minimax(new_board, depth-1, True, mark, config, alpha, beta))
                if value <= alpha:
                    break
                beta = np.minimum(beta, value)
            return value
    def score_move(grid, col, mark, config, nsteps):
        next_grid = drop_piece(grid, col, mark, config)
        score = minimax(next_grid, nsteps-1, False, mark, config, -np.inf, np.inf)
        return score

    N_STEPS = 5
    valid_moves = [c for c in range(config.columns) if obs.board[c] == 0]
    grid = np.asarray(obs.board).reshape(config.rows, config.columns)
    scores = dict(zip(valid_moves, [score_move(grid, col, obs.mark, config, N_STEPS) for col in valid_moves]))
    max_cols = [key for key in scores.keys() if scores[key] == max(scores.values())]
    return random.choice(max_cols)

In [82]:
import inspect
def write_agent_to_file(function, file):
    with open(file, "w") as f:
        f.write(inspect.getsource(function))
        print(function, "written to", file)

write_agent_to_file(agent_minimax, 'minimax_agent.py')

<function agent_minimax at 0x7fe190e709a0> written to minimax_agent.py


In [83]:
import sys
from kaggle_environments import evaluate, make, utils, agent
out = sys.stdout
submission = utils.read_file('minimax_agent.py')
saved_agent = agent.get_last_callable(submission)
sys.stdout = out

env = make("connectx", debug=True)
env.run([saved_agent, saved_agent])
print("Success!" if env.state[0].status == env.state[1].status == "DONE" else "Failed...")

Success!
